# R.O.M. — Fully Unitless Formulation: Comprehensive Test

**Claim under test (factorization hypothesis).** Every quantity $X$ of the closed
R.O.M. algebraic system factorizes as

$$X \;=\; \hat X(\beta, e;\, O;\, i, \omega_i)\;\cdot\; S_X(R_s;\, c,\, G)$$

where $\hat X$ is a closed-form **pure number** computed from dimensionless relational
inputs only, and $S_X$ is a monomial **scale factor** (powers of $R_s$, $c$, $G$).
Physical units, $c$ and $G$ enter *only* through $S_X$ — as the final scaling of an
already fully solved relational system.

**Derivation contract (methodological constraints).**
- Allowed inputs: the closed R.O.M. system exactly as documented in
  *R.O.M. — Closed Algebraical System* (all 66 expressions verified closed in
  `ROM_FULL_TEST.ipynb`), plus the observational-inversion formulas of
  §*Relational Parameterization* and §*Algebraic Determination of Absolute System Scale*.
- No GR machinery is invoked anywhere; no external dynamical law is imported.
- Every claimed cancellation ($c$-independence, $G$-independence, unit-system
  independence) is a **computed output** of this notebook, not an assumption.
- Real-data comparison (Mercury, Jupiter) comes **last**, as an application.

**Irreducible primitives** (structural, all dimensionless):
1. $S^1$ kinematic carrier: $\beta^2 + \beta_Y^2 = 1$
2. $S^2$ potential carrier: $\kappa^2 + \kappa_X^2 = 1$
3. Closure Theorem: $\kappa^2 = 2\beta^2$
4. Energy Symmetry Law $\Delta E_{A\to B} + \Delta E_{B\to A} = 0$, whose orbital
   form is the global invariant $\beta^2 = \kappa_o^2 - \beta_o^2$

**Intrinsic state — necessary and sufficient inputs:**
- $(\beta, e)$: two pure numbers fix the entire global orbit state;
- $+\,O$: one phase coordinate for local (phase-resolved) states;
- $+\,(i, \omega_i)$: two orientation angles for observer projections;
- $+\,\kappa_R$ (optional): central-body surface channel, for $R_{sur}$, $\tau_D$, $\theta_{sur}$;
- $+$ exactly **one** dimensional anchor (one clock or ruler reading) *iff* outputs in
  local units are desired. $c$ converts the observer's time unit into their length unit;
  $G$ appears only when converting the geometric mass $R_s/2$ into kilograms.

In geometric units every system has $\hat M = 1/2$: mass is not a parameter of the
relational state — it *is* the scale.

Precision: `mpmath` with 60 significant digits; identities must close to $10^{-50}$.

> Runs as-is on Google Colab (`mpmath` is preinstalled). For a local run: `pip install mpmath`. Total runtime: a few seconds.

In [1]:
# ------------------------------------------------------------------
# Cell 1 — Precision, helpers, reproducible sampling
# ------------------------------------------------------------------
import random, inspect
import mpmath as mp

mp.mp.dps = 60            # 60-digit working precision
TOL = mp.mpf('1e-50')     # identities must close to 50 digits
PI = mp.pi

random.seed(20260704)     # fixed seed: every run reproduces the same draws

RESULTS = []              # global scoreboard: (test_block, check_name, passed, worst_error)

def rel_err(x, y):
    """Relative error |x - y| / max(|x|,|y|,1). Pure-number comparison."""
    x, y = mp.mpf(x), mp.mpf(y)
    scale = max(abs(x), abs(y), mp.mpf(1))
    return abs(x - y) / scale

def check(block, name, x, y, tol=TOL):
    """Record whether two independently computed pure numbers agree to tolerance."""
    err = rel_err(x, y)
    RESULTS.append((block, name, err <= tol, err))
    return err <= tol

def rnd(lo, hi):
    """Reproducible random mpf uniform in [lo, hi] with 18-digit entropy."""
    u = mp.mpf(random.randint(0, 10**18)) / mp.mpf(10**18)
    return mp.mpf(lo) + (mp.mpf(hi) - mp.mpf(lo)) * u

def draw_state():
    """Draw a random admissible intrinsic state (beta, e).
    Domain: bound, sub-horizon orbit — keep kappa_p^2 = 2 beta^2/(1-e) <= 0.9
    so every projection stays real all the way to perihelion."""
    e = rnd('0', '0.95')
    beta_max = min(mp.mpf('0.35'), mp.sqrt(mp.mpf('0.45') * (1 - e)))
    beta = rnd('0.001', beta_max)
    return beta, e

print(f"mpmath precision: {mp.mp.dps} digits, tolerance {mp.nstr(TOL, 3)}")

mpmath precision: 60 digits, tolerance 1.0e-50


## The Intrinsic Core — unitless solver

The three functions below are the *entire* relational solution. They accept only pure
numbers and return only pure numbers. Hatted quantities are expressed in the system's
own geometric units: lengths in units of $R_s$, times in units of $R_s/c$, frequencies
in units of $c/R_s$, specific angular momentum in units of $R_s c$, densities in units
of $c^2/(G R_s^2)$. Setting $R_s = 1, c = 1$ makes the hats disappear.

By construction the source code of the core contains **no** $c$, **no** $G$, **no**
unit conversion of any kind — Cell 3 asserts this mechanically.

In [2]:
# ------------------------------------------------------------------
# Cell 2 — THE INTRINSIC CORE (pure numbers in, pure numbers out)
# ------------------------------------------------------------------

def core_global(beta, e, kappa_R=None):
    """Global block: the full orbit-invariant state from the intrinsic pair (beta, e).
    Optional kappa_R (surface potential projection of the central body) opens the
    surface channel: R_sur, tau_D, theta_sur. Everything is a pure number."""
    s = {}
    s['beta'] = beta
    s['e'] = e

    # --- primitives on the two carriers -----------------------------------
    s['kappa']   = beta * mp.sqrt(2)               # Closure Theorem  kappa^2 = 2 beta^2
    s['beta_Y']  = mp.sqrt(1 - beta**2)            # S^1 conservation
    s['kappa_X'] = mp.sqrt(1 - s['kappa']**2)      # S^2 conservation
    s['theta_1'] = mp.acos(beta)                   # distribution angle on S^1
    s['theta_2'] = mp.asin(s['kappa'])             # distribution angle on S^2

    # --- relational spacetime factors and total shift ----------------------
    s['tau']   = s['kappa_X'] * s['beta_Y']        # relational spacetime factor
    s['tau_Y'] = mp.sqrt(1 - s['tau']**2)          # its phase factor = sqrt(3b^2-2b^4)
    s['Q']     = mp.sqrt(s['kappa']**2 + beta**2)  # total relational shift = sqrt(3) beta
    s['Q_Y']   = mp.sqrt(1 - s['Q']**2)            # orthogonal complement of Q
    s['Z_sys'] = 1 / s['tau']                      # combined spectroscopic shift
    s['z_k']   = 1/s['kappa_X'] - 1                # gravitational redshift at a
    s['z_b']   = 1/s['beta_Y'] - 1                 # transverse Doppler at a

    # --- eccentricity geometry --------------------------------------------
    s['e_Y'] = mp.sqrt(1 - e**2)                   # orthogonal eccentricity value
    s['e_X'] = (1 + e) / (1 - e)                   # shape factor
    s['beta_int'] = beta / s['e_Y']                # interior kinetic projection

    # --- geometric scale in the system's own units (R_s = 1) ---------------
    s['a_hat']   = 1 / s['kappa']**2               # semi-major axis  = 1/(2 beta^2)
    s['r_p_hat'] = s['a_hat'] * (1 - e)            # perihelion radius
    s['r_a_hat'] = s['a_hat'] * (1 + e)            # aphelion radius
    s['t_hat']   = s['a_hat']                      # light-time scale a/c in units R_s/c

    # --- apsidal projections ------------------------------------------------
    s['kappa_p']  = s['kappa'] / mp.sqrt(1 - e)    # potential projection at perihelion
    s['kappa_Xp'] = mp.sqrt(1 - s['kappa_p']**2)   # its phase factor
    s['beta_p']   = beta * mp.sqrt((1 + e)/(1 - e))# kinetic projection at perihelion
    s['delta_p']  = 1 / (1 + e)                    # closure factor at perihelion
    s['Q_p']      = mp.sqrt(s['kappa_p']**2 + s['beta_p']**2)
    s['beta_a']   = beta * mp.sqrt((1 - e)/(1 + e))# kinetic projection at aphelion
    s['kappa_a']  = mp.sqrt(beta**2 + s['beta_a']**2)
    s['delta_a']  = 1 / (1 - e)                    # closure factor at aphelion
    s['Q_a']      = mp.sqrt(s['kappa_a']**2 + s['beta_a']**2)

    # --- temporal structure in the system's own units (R_s/c = 1) ----------
    s['omega_hat'] = 2 * beta**3                   # angular frequency
    s['T_hat']     = PI / beta**3                  # orbital period
    s['h_hat']     = s['e_Y'] / (2 * beta)         # invariant angular momentum (R_s c = 1)

    # --- exact second-order precession --------------------------------------
    s['Omega_phi'] = s['tau_Y']**2 / (1 - e**2)    # precession per phase radian
    s['Delta_phi'] = 2 * PI * s['Omega_phi']       # precession per orbit
    s['Omega']     = 1 - s['Omega_phi']            # true/idealised phase ratio O/o

    # --- balance points (r = a, local closure kappa_o^2 = 2 beta_o^2) ------
    s['B_a'] = mp.acos(-e)                         # ascending balance point
    s['B_d'] = 2*PI - mp.acos(-e)                  # descending balance point

    # --- density channel (units c^2/(G R_s^2) = 1) --------------------------
    s['rho_hat']    = s['kappa']**6 / (8*PI)       # relational density
    s['rho_max_hat']= s['kappa']**4 / (8*PI)       # horizon bound at this scale

    # --- geometric mass: the universal constant 1/2 -------------------------
    s['M_hat'] = mp.mpf(1)/2                       # mass in units c^2 R_s / G — every system

    # --- optional surface channel (central body structure) ------------------
    if kappa_R is not None:
        s['kappa_R']     = kappa_R
        s['R_sur_hat']   = 1 / kappa_R**2                    # physical surface radius
        s['sin_theta_sys'] = s['kappa']**2 / kappa_R**2      # angular radius from orbit
        s['tau_D_hat']   = 2*mp.sqrt(2)*PI / kappa_R**3      # time-density invariant
    return s


def core_phase(beta, e, O):
    """Phase block: the local relational state at true phase O (pure numbers)."""
    g = core_global(beta, e)
    p = {}
    p['O'] = O
    p['eta_o']    = (1 - e**2) / (1 + e*mp.cos(O))           # r/a — normalised radius
    p['r_o_hat']  = g['a_hat'] * p['eta_o']                  # radius in units of R_s
    p['kappa_o']  = mp.sqrt(1 / p['r_o_hat'])                # local potential projection
    p['kappa_Xo'] = mp.sqrt(1 - p['kappa_o']**2)             # local potential phase factor
    p['beta_o']   = mp.sqrt(p['kappa_o']**2 - beta**2)       # binding-energy invariant
    p['beta_Yo']  = mp.sqrt(1 - p['beta_o']**2)              # local kinetic phase factor
    p['beta_T']   = beta * (1 + e*mp.cos(O)) / g['e_Y']      # transverse kinetic projection
    p['beta_R']   = beta * e*mp.sin(O) / g['e_Y']            # radial kinetic projection
    p['delta_o']  = p['kappa_o']**2 / (2*p['beta_o']**2)     # local closure factor
    p['Q_o']      = mp.sqrt(p['kappa_o']**2 + p['beta_o']**2)# local relational shift
    p['Delta_Q']  = p['Q_o']**2 - g['Q']**2                  # phase shift amplitude
    p['omega_o_hat'] = 2*beta**3 * (1 + e*mp.cos(O))**2 / g['e_Y']**3  # local frequency
    p['tau_o']    = p['kappa_Xo'] * p['beta_Yo']             # phase spacetime factor
    p['tau_Yo']   = mp.sqrt(1 - p['tau_o']**2)               # its amplitude factor
    p['Z_sys_O']  = 1 / p['tau_o']                           # combined shift at phase O
    p['z_ko']     = 1/p['kappa_Xo'] - 1                      # gravitational redshift at O
    p['z_bo']     = 1/p['beta_Yo'] - 1                       # transverse Doppler at O
    p['t_o_hat']  = p['r_o_hat']                             # light-time at O (R_s/c = 1)
    p['zeta']     = zeta_closed(e, O)                        # temporal phase interval
    p['Delta_to_hat'] = p['zeta'] / (2*beta**3)              # duration of [0, O] (R_s/c = 1)
    return p


def zeta_closed(e, O):
    """Temporal phase interval zeta (mean-anomaly analog), closed form.
    Continuous atan2 realisation of the documented two-branch arctan formula:
    identical on the ascending branch, and automatically carries the +2*pi of
    the descending branch. Valid for O in [0, 2*pi)."""
    e_X = (1 + e) / (1 - e)
    e_Y = mp.sqrt(1 - e**2)
    E_half = mp.atan2(mp.sin(O/2), mp.sqrt(e_X) * mp.cos(O/2))
    return 2*E_half - e*e_Y*mp.sin(O) / (1 + e*mp.cos(O))


def core_observer(beta, e, O, i, omega_i):
    """Observer block: sky-projected pure numbers for orientation (i, omega_i)."""
    g = core_global(beta, e)
    p = core_phase(beta, e, O)
    o = {}
    o['K_i']    = g['beta_int'] * mp.sin(i)                            # semi-amplitude invariant
    o['beta_z'] = g['beta_int'] * (mp.cos(O + omega_i) + e*mp.cos(omega_i)) * mp.sin(i)
    o['Z_raw']  = (1 + o['beta_z']) * p['Z_sys_O']                     # raw light shift
    o['B_ai']   = g['B_a'] + omega_i                                   # shifted balance point
    o['x_sky_hat']   = p['r_o_hat'] * mp.cos(O + omega_i)              # sky-plane coordinates
    o['y_sky_hat']   = p['r_o_hat'] * mp.sin(O + omega_i) * mp.cos(i)  # (units of R_s)
    o['z_depth_hat'] = p['r_o_hat'] * mp.sin(O + omega_i) * mp.sin(i)
    return o


# --- Observational inversion routes: pure ratios -> intrinsic state --------
# e-channel (orbit shape) — any ONE of:
def e_from_omega_ratio(w_ratio):
    """E1: ratio of fastest to slowest apparent angular speed."""
    return (mp.sqrt(w_ratio) - 1) / (mp.sqrt(w_ratio) + 1)

def e_from_apsidal_doppler(beta_p, beta_a):
    """E2: apsidal Doppler pair."""
    return (beta_p - beta_a) / (beta_p + beta_a)

def e_from_apsidal_radii(ra_over_rp):
    """E3: apsidal astrometric ratio (an angle ratio on the sky)."""
    return (ra_over_rp - 1) / (ra_over_rp + 1)

def e_from_balance_phase(B_a):
    """E4: phase location of the ascending balance point."""
    return -mp.cos(B_a)

# beta-channel (orbit depth) — any ONE of:
def beta_from_zb(z_b):
    """B1: transverse Doppler shift at the semi-major axis."""
    return mp.sqrt(1 - (1 + z_b)**-2)

def beta_from_zk(z_k):
    """B2: gravitational redshift at the semi-major axis (via Closure)."""
    return mp.sqrt((1 - (1 + z_k)**-2) / 2)

def beta_from_Zsys(Z_sys):
    """B3: combined shift Z_sys = 1/tau at the semi-major axis (or balance point).
    Inverts tau^2 = (1-2b^2)(1-b^2): the same quadratic as the balance-point theorem."""
    tau2 = 1 / Z_sys**2
    return mp.sqrt((3 - mp.sqrt(1 + 8*tau2)) / 4)

def beta_from_apsidal_doppler(beta_p, beta_a):
    """B4: apsidal Doppler pair — geometric mean."""
    return mp.sqrt(beta_p * beta_a)

def beta_from_two_point(r2_over_r1, beta_1, beta_2):
    """B5: two arbitrary trajectory points (astrometric ratio + two Dopplers).
    Vis-viva invariant: kappa_1^2 - kappa_2^2 = beta_1^2 - beta_2^2."""
    kappa1_sq = (beta_1**2 - beta_2**2) * r2_over_r1 / (r2_over_r1 - 1)
    return mp.sqrt(kappa1_sq - beta_1**2)

def beta_from_surface(z_surf, sin_theta_obs):
    """B6: surface channel — surface redshift + angular radius seen from the orbit.
    kappa^2 = kappa_R^2 sin(theta); beta^2 = kappa^2/2."""
    kappa_R_sq = 1 - (1 + z_surf)**-2
    return mp.sqrt(kappa_R_sq * sin_theta_obs / 2)

def kappa_sq_from_tau_balance(tau_Ba):
    """Method C (Balance Point): kappa^2 = (3 - sqrt(1+8 tau^2))/2 from the light
    signal tau measured at the balance point r = a."""
    return (3 - mp.sqrt(1 + 8*tau_Ba**2)) / 2

def Rs_hat_arbitrary_phase(a_hat, r_hat, tau_o):
    """Method D (Arbitrary Phase): recover the scale from single-epoch (a, r, tau_o).
    In intrinsic units the recovered scale must equal exactly 1."""
    disc = (4*a_hat - r_hat)**2 - 8*a_hat*(2*a_hat - r_hat)*(1 - tau_o**2)
    return r_hat * (4*a_hat - r_hat - mp.sqrt(disc)) / (2*(2*a_hat - r_hat))

def dE_ledger(kX_A, bY_A, kX_B, bY_B):
    """Energy Symmetry Law ledger entry Delta E_{A->B} / E_0."""
    return kX_B/bY_B - kX_A/bY_A

print("Intrinsic core loaded: core_global, core_phase, core_observer + 6 beta-routes, 4 e-routes.")

Intrinsic core loaded: core_global, core_phase, core_observer + 6 beta-routes, 4 e-routes.


In [3]:
# ------------------------------------------------------------------
# Cell 3 — Structural assertion: the core contains no dimensional constant
# ------------------------------------------------------------------
# The dimensional constants used later in this notebook are ONLY ever named
# c_light and G_newton. The intrinsic core must not reference them, nor any
# numerical value of c or G. This is checked mechanically on the source code.

CORE_FUNCS = [core_global, core_phase, zeta_closed, core_observer,
              e_from_omega_ratio, e_from_apsidal_doppler, e_from_apsidal_radii,
              e_from_balance_phase, beta_from_zb, beta_from_zk, beta_from_Zsys,
              beta_from_apsidal_doppler, beta_from_two_point, beta_from_surface,
              kappa_sq_from_tau_balance, Rs_hat_arbitrary_phase, dE_ledger]

core_source = "".join(inspect.getsource(f) for f in CORE_FUNCS)
forbidden = ['c_light', 'G_newton', '299792458', '2.998', '6.674', '6.6743']
leaks = [tok for tok in forbidden if tok in core_source]
assert not leaks, f"Dimensional constants leaked into the intrinsic core: {leaks}"
print("STRUCTURAL CHECK PASSED: the intrinsic core references no c, no G, no unit constant.")
print(f"  ({len(CORE_FUNCS)} functions, {len(core_source.splitlines())} source lines scanned)")

STRUCTURAL CHECK PASSED: the intrinsic core references no c, no G, no unit constant.
  (17 functions, 170 source lines scanned)


## TEST 1 — Intrinsic closure of the unitless core

Random admissible states $(\beta, e)$, random phases $O$, random orientations
$(i, \omega_i)$. Every multi-form identity of the documented system is evaluated
through **algebraically independent paths** and must agree to $10^{-50}$.
No dimensional quantity exists anywhere in this test.

In [4]:
# ------------------------------------------------------------------
# Cell 4 — TEST 1: identity battery (pure numbers only)
# ------------------------------------------------------------------
N_DRAWS = 120
N_QUAD  = 12   # subset for the (slow) 50-digit numerical quadrature cross-check

for n in range(N_DRAWS):
    beta, e = draw_state()
    O  = rnd('0.02', str(float(2*mp.pi) - 0.02))   # keep clear of exact 0, 2*pi
    i  = rnd('0.05', str(float(mp.pi)/2 - 0.05))
    wi = rnd('0', str(float(2*mp.pi)))
    kappa_R = rnd(str(float(mp.sqrt(2)*beta)*1.05), '0.99')   # surface deeper than orbit
    g = core_global(beta, e, kappa_R=kappa_R)
    p = core_phase(beta, e, O)
    obs = core_observer(beta, e, O, i, wi)

    # -- carriers and closure ------------------------------------------------
    check(1, 'S1 carrier: beta^2+beta_Y^2 = 1', g['beta']**2 + g['beta_Y']**2, 1)
    check(1, 'S2 carrier: kappa^2+kappa_X^2 = 1', g['kappa']**2 + g['kappa_X']**2, 1)
    check(1, 'closure: kappa^2 = 2 beta^2', g['kappa']**2, 2*beta**2)
    check(1, 'redshift inversion recovers kappa', 1 - (1+g['z_k'])**-2, g['kappa']**2)
    check(1, 'redshift inversion recovers beta',  1 - (1+g['z_b'])**-2, beta**2)
    check(1, 'theta_1: sin(theta_1) = beta_Y', mp.sin(g['theta_1']), g['beta_Y'])
    check(1, 'theta_2: cos(theta_2) = kappa_X', mp.cos(g['theta_2']), g['kappa_X'])

    # -- spacetime factors and total shift ------------------------------------
    check(1, 'tau = kappa_X beta_Y = 1/Z_sys', g['tau'], 1/g['Z_sys'])
    check(1, 'Z_sys = (1+z_k)(1+z_b)', g['Z_sys'], (1+g['z_k'])*(1+g['z_b']))
    check(1, 'tau_Y^2 = 3 beta^2 - 2 beta^4', g['tau_Y']**2, 3*beta**2 - 2*beta**4)
    check(1, 'Q = sqrt(3) beta', g['Q'], mp.sqrt(3)*beta)
    check(1, 'Q = sqrt(3/2) kappa', g['Q'], mp.sqrt(mp.mpf(3)/2)*g['kappa'])
    check(1, 'Q_Y^2 = 1 - 3 beta^2', g['Q_Y']**2, 1 - 3*beta**2)

    # -- geometric scale (hats: R_s = 1) --------------------------------------
    check(1, 'a_hat = 1/kappa^2', g['a_hat'], 1/g['kappa']**2)
    check(1, 'binding energy beta^2 = 1/(2 a_hat)', beta**2, 1/(2*g['a_hat']))
    check(1, 'r_p_hat = 1/kappa_p^2', g['r_p_hat'], 1/g['kappa_p']**2)
    check(1, 'r_a_hat = 1/kappa_a^2', g['r_a_hat'], 1/g['kappa_a']**2)

    # -- apsidal projections and closure factors ------------------------------
    check(1, 'kappa_p^2 = 2 beta^2/(1-e)', g['kappa_p']**2, 2*beta**2/(1-e))
    check(1, 'beta_p^2 = kappa_p^2 (1+e)/2', g['beta_p']**2, g['kappa_p']**2*(1+e)/2)
    check(1, 'delta_p = kappa_p^2/(2 beta_p^2)', g['delta_p'], g['kappa_p']**2/(2*g['beta_p']**2))
    check(1, 'delta_a = kappa_a^2/(2 beta_a^2)', g['delta_a'], g['kappa_a']**2/(2*g['beta_a']**2))
    check(1, 'kappa_a^2 = beta^2 + beta_a^2', g['kappa_a']**2, beta**2 + g['beta_a']**2)
    check(1, 'e = 2 beta_p^2/kappa_p^2 - 1', e, 2*g['beta_p']**2/g['kappa_p']**2 - 1)
    check(1, 'e = 1 - 2 beta_a^2/kappa_a^2', e, 1 - 2*g['beta_a']**2/g['kappa_a']**2)
    check(1, 'e = 1/delta_p - 1', e, 1/g['delta_p'] - 1)

    # -- structural-dynamical equivalence chain (shape factor e_X) ------------
    eX = g['e_X']
    check(1, 'e_X = r_a/r_p', eX, g['r_a_hat']/g['r_p_hat'])
    check(1, 'e_X = delta_a/delta_p', eX, g['delta_a']/g['delta_p'])
    check(1, 'e_X = beta_p/beta_a', eX, g['beta_p']/g['beta_a'])
    check(1, 'e_X = kappa_p^2/kappa_a^2', eX, g['kappa_p']**2/g['kappa_a']**2)
    check(1, 'e_X = kappa_a^2 beta_p^2/(kappa_p^2 beta_a^2)', eX,
          g['kappa_a']**2*g['beta_p']**2/(g['kappa_p']**2*g['beta_a']**2))

    # -- precession: three documented forms ------------------------------------
    check(1, 'Delta_phi form tau_Y vs beta', g['Delta_phi'], 2*PI*(3*beta**2-2*beta**4)/(1-e**2))
    check(1, 'Delta_phi form R_s/a (hats)', g['Delta_phi'],
          3*PI/(g['a_hat']*(1-e**2)) - PI/(g['a_hat']**2*(1-e**2)))

    # -- temporal structure (hats: R_s/c = 1) -----------------------------------
    check(1, 'T_hat omega_hat = 2 pi', g['T_hat']*g['omega_hat'], 2*PI)
    check(1, 'Kepler ratio: T_hat = 2 sqrt(2) pi a_hat^(3/2)', g['T_hat'],
          2*mp.sqrt(2)*PI*g['a_hat']**mp.mpf('1.5'))
    check(1, 'beta = 2 pi a_hat / T_hat', beta, 2*PI*g['a_hat']/g['T_hat'])
    check(1, 'beta = (pi/T_hat)^(1/3)', beta, (PI/g['T_hat'])**(mp.mpf(1)/3))
    check(1, 'h_hat = a_hat beta e_Y', g['h_hat'], g['a_hat']*beta*g['e_Y'])

    # -- density channel and surface channel -------------------------------------
    check(1, 'kappa^2 = rho/rho_max', g['kappa']**2, g['rho_hat']/g['rho_max_hat'])
    check(1, 'tau_D_hat = T_hat sin(theta_sys)^(3/2)', g['tau_D_hat'],
          g['T_hat']*g['sin_theta_sys']**mp.mpf('1.5'))
    check(1, 'sin(theta_sys) = kappa^2/kappa_R^2', g['sin_theta_sys'],
          g['kappa']**2/g['kappa_R']**2)
    check(1, 'R_sur_hat = a_hat sin(theta_sys)', g['R_sur_hat'],
          g['a_hat']*g['sin_theta_sys'])

    # -- phase block ---------------------------------------------------------------
    check(1, 'eta_o = r_o/a', p['eta_o'], p['r_o_hat']/g['a_hat'])
    check(1, 'kappa_o^2 = 1/r_o_hat', p['kappa_o']**2, 1/p['r_o_hat'])
    check(1, 'kappa_o = kappa_p sqrt((1+e cos O)/(1+e))', p['kappa_o'],
          g['kappa_p']*mp.sqrt((1+e*mp.cos(O))/(1+e)))
    check(1, 'eta_o = 2 - 2 beta_o^2/kappa_o^2', p['eta_o'],
          2 - 2*p['beta_o']**2/p['kappa_o']**2)
    check(1, 'binding: beta^2 = kappa_o^2 - beta_o^2', beta**2,
          p['kappa_o']**2 - p['beta_o']**2)
    check(1, 'phase invariant beta^2-beta_o^2 = kappa^2-kappa_o^2',
          beta**2 - p['beta_o']**2, g['kappa']**2 - p['kappa_o']**2)
    check(1, 'beta_o^2 = beta_R^2 + beta_T^2', p['beta_o']**2,
          p['beta_R']**2 + p['beta_T']**2)
    check(1, 'beta_o alt form (1+e^2+2e cos O)', p['beta_o'],
          beta*mp.sqrt(1+e**2+2*e*mp.cos(O))/g['e_Y'])
    check(1, 'beta_o = kappa_o/sqrt(2 delta_o)', p['beta_o'],
          p['kappa_o']/mp.sqrt(2*p['delta_o']))
    check(1, 'beta_o^2 = 1/r_o_hat - 1/(2 a_hat)', p['beta_o']**2,
          1/p['r_o_hat'] - 1/(2*g['a_hat']))
    check(1, 'delta_o algebraic form', p['delta_o'],
          (1+e*mp.cos(O))/(1+e**2+2*e*mp.cos(O)))
    check(1, 'beta_T = kappa_o^2 e_Y/(2 beta)', p['beta_T'],
          p['kappa_o']**2*g['e_Y']/(2*beta))
    check(1, 'beta_T/kappa_o^2 = e_Y/(2 beta)  [phase invariant]',
          p['beta_T']/p['kappa_o']**2, g['e_Y']/(2*beta))
    check(1, 'beta_T^2 eta_o^2 = beta^2 (1-e^2)  [phase invariant]',
          p['beta_T']**2*p['eta_o']**2, beta**2*(1-e**2))
    check(1, 'h_hat = r_o_hat^2 omega_o_hat  [phase invariant]',
          g['h_hat'], p['r_o_hat']**2*p['omega_o_hat'])
    check(1, 'omega_o_hat = h_hat/r_o_hat^2 dual form', p['omega_o_hat'],
          g['a_hat']*beta*g['e_Y']/p['r_o_hat']**2)
    check(1, 'tau_o = 1/Z_sys(O)', p['tau_o'], 1/p['Z_sys_O'])
    check(1, 'phase redshift inversion kappa_o', 1-(1+p['z_ko'])**-2, p['kappa_o']**2)
    check(1, 'phase redshift inversion beta_o',  1-(1+p['z_bo'])**-2, p['beta_o']**2)
    check(1, 'Q_o^2 = kappa_o^2 + beta_o^2', p['Q_o']**2, p['kappa_o']**2+p['beta_o']**2)
    check(1, 'Delta_Q = 2(kappa_o^2 - kappa^2)', p['Delta_Q'],
          2*(p['kappa_o']**2 - g['kappa']**2))

    # -- Energy Symmetry Law: ledger closes between two random phases -----------
    O_B = rnd('0.02', str(float(2*mp.pi) - 0.02))
    pB = core_phase(beta, e, O_B)
    dAB = dE_ledger(p['kappa_Xo'], p['beta_Yo'], pB['kappa_Xo'], pB['beta_Yo'])
    dBA = dE_ledger(pB['kappa_Xo'], pB['beta_Yo'], p['kappa_Xo'], p['beta_Yo'])
    check(1, 'Energy Symmetry Law: dE(A->B)+dE(B->A) = 0', dAB + dBA, 0)

    # -- balance points -----------------------------------------------------------
    pb = core_phase(beta, e, g['B_a'])
    check(1, 'balance point: r = a', pb['r_o_hat'], g['a_hat'])
    check(1, 'balance point: local closure kappa_o^2 = 2 beta_o^2',
          pb['kappa_o']**2, 2*pb['beta_o']**2)
    check(1, 'balance point: delta_o = 1', pb['delta_o'], 1)
    check(1, 'balance point: tau_o(B_a) = global tau', pb['tau_o'], g['tau'])
    pd = core_phase(beta, e, g['B_d'])
    check(1, 'zeta balance symmetry B_a - zeta(B_a) = zeta(B_d) - B_d',
          g['B_a'] - pb['zeta'], pd['zeta'] - g['B_d'])

    # -- Method C: recover kappa^2 from the balance-point light signal -----------
    check(1, 'Method C: kappa^2 from tau(B_a) quadratic',
          kappa_sq_from_tau_balance(pb['tau_o']), g['kappa']**2)

    # -- Method D: single-epoch scale recovery must return exactly 1 -------------
    check(1, 'Method D: R_s_hat from (a_hat, r_hat, tau_o) = 1',
          Rs_hat_arbitrary_phase(g['a_hat'], p['r_o_hat'], p['tau_o']), 1)

    # -- observer block -------------------------------------------------------------
    check(1, 'K_i = beta_int sin(i)', obs['K_i'], g['beta_int']*mp.sin(i))
    check(1, 'sky norm: x^2+y^2+z^2 = r_o^2',
          obs['x_sky_hat']**2 + obs['y_sky_hat']**2 + obs['z_depth_hat']**2,
          p['r_o_hat']**2)

    # -- documented closed forms of Z_sys at the extremal phases -----------------
    for sign, Oext, tag in [(+1, -wi, 'Z_sys(-omega_i)'), (-1, PI - wi, 'Z_sys(pi-omega_i)')]:
        t1 = (1 - beta**2*(1+e**2+2*sign*e*mp.cos(wi))/(1-e**2))**mp.mpf('-0.5')
        t2 = (1 - 2*beta**2*(1+sign*e*mp.cos(wi))/(1-e**2))**mp.mpf('-0.5')
        pext = core_phase(beta, e, Oext)
        check(1, f'{tag} closed form = 1/tau_o', t1*t2, pext['Z_sys_O'])

    # -- Z_raw extremes and the Holographic Decryption Invariant ------------------
    p1 = core_phase(beta, e, -wi)
    p2 = core_phase(beta, e, PI - wi)
    o1 = core_observer(beta, e, -wi, i, wi)
    o2 = core_observer(beta, e, PI - wi, i, wi)
    check(1, 'Z_rawmax = Z_sys(-omega_i)(1+K_i(1+e cos omega_i))', o1['Z_raw'],
          p1['Z_sys_O']*(1 + obs['K_i']*(1 + e*mp.cos(wi))))
    check(1, 'Z_rawmin = Z_sys(pi-omega_i)(1+K_i(-1+e cos omega_i))', o2['Z_raw'],
          p2['Z_sys_O']*(1 + obs['K_i']*(-1 + e*mp.cos(wi))))
    holo = (o1['Z_raw']*p1['tau_o']*(1 - e*mp.cos(wi))
            + o2['Z_raw']*p2['tau_o']*(1 + e*mp.cos(wi)))
    check(1, 'HOLOGRAPHIC DECRYPTION INVARIANT = 2', holo, 2)

    # -- zeta: closed form vs documented two-branch arctan form -------------------
    e_Xv, e_Yv = g['e_X'], g['e_Y']
    branch = (2*mp.atan(mp.sin(O/2)/(mp.sqrt(e_Xv)*mp.cos(O/2)))
              - e*e_Yv*mp.sin(O)/(1+e*mp.cos(O)))
    if O > PI:
        branch += 2*PI
    check(1, 'zeta: atan2 form = documented branch form', p['zeta'], branch)
    check(1, 'Delta_to_hat = zeta T_hat/(2 pi)', p['Delta_to_hat'],
          p['zeta']*g['T_hat']/(2*PI))

    # -- zeta vs 50-digit numerical quadrature (subset: slow) ---------------------
    if n < N_QUAD:
        integral = mp.quad(lambda th: (1 + e*mp.cos(th))**-2, [0, min(O, PI), O] if O > PI else [0, O])
        zeta_int = (1 - e**2)**mp.mpf('1.5') * integral
        check(1, 'zeta: closed form = integral (mp.quad)', p['zeta'], zeta_int,
              tol=mp.mpf('1e-45'))

# -- critical relational states (documented table, exact rationals) --------------
for r_hat, k2, b2, Q2 in [(3, mp.mpf(1)/3, mp.mpf(1)/6, mp.mpf(1)/2),
                          (mp.mpf(3)/2, mp.mpf(2)/3, mp.mpf(1)/3, 1),
                          (1, 1, mp.mpf(1)/2, mp.mpf(3)/2)]:
    check(1, f'critical state r={mp.nstr(mp.mpf(r_hat),3)}R_s: kappa_loc^2', mp.mpf(1)/r_hat, k2)
    check(1, f'critical state r={mp.nstr(mp.mpf(r_hat),3)}R_s: closure beta^2', k2/2, b2)
    check(1, f'critical state r={mp.nstr(mp.mpf(r_hat),3)}R_s: Q^2 = 3 beta^2', 3*b2, Q2)

n1 = [r for r in RESULTS if r[0] == 1]
print(f"TEST 1: {sum(1 for r in n1 if r[2])}/{len(n1)} identity evaluations passed "
      f"({N_DRAWS} random states, worst error {mp.nstr(max(r[3] for r in n1), 3)})")

TEST 1: 9501/9501 identity evaluations passed (120 random states, worst error 3.91e-57)


## TEST 2 — Factorization: solve-then-scale $\equiv$ dimensional pipeline

A fully **dimensional** pipeline (SI formulas transcribed verbatim from the document,
with explicit $c$ and $G$) is compared against the **unitless core scaled at the end**
by the single anchor $R_s$. Anchors span 41 orders of magnitude — from subatomic to
galactic — with the same intrinsic algebra. Agreement to 50 digits everywhere means:
the diagram commutes, units are pure bookkeeping.

In [5]:
# ------------------------------------------------------------------
# Cell 5 — TEST 2: commutation of solving and scaling
# ------------------------------------------------------------------
c_light  = mp.mpf('299792458')       # SI values — used ONLY outside the core
G_newton = mp.mpf('6.6743e-11')

def dimensional_pipeline(c, G, M, a, e, O, kappa_R=None):
    """Literal transcription of the DIMENSIONAL formulas from the document.
    Independent of the intrinsic core: everything carries units here."""
    d = {}
    R_s   = 2*G*M/c**2                          # Schwarzschild radius (unit translation)
    kappa = mp.sqrt(R_s/a)
    beta  = kappa/mp.sqrt(2)
    e_Y   = mp.sqrt(1-e**2)
    d['R_s']   = R_s
    d['a']     = a
    d['r_p']   = a*(1-e)
    d['r_a']   = a*(1+e)
    d['omega'] = beta*c/a                       # angular frequency  [1/s]
    d['T']     = 2*PI*a/(beta*c)                # orbital period     [s]
    d['h_W']   = a*beta*c*e_Y                   # angular momentum   [m^2/s]
    d['M']     = kappa**2*c**2*a/(2*G)          # mass parameter     [kg]
    d['rho']   = kappa**2*c**2/(8*PI*G*a**2)    # relational density [kg/m^3]
    d['t']     = a/c                            # temporal scale     [s]
    d['r_o']     = a*(1-e**2)/(1+e*mp.cos(O))   # radius at phase    [m]
    d['omega_o'] = (beta*c/a)*(1+e*mp.cos(O))**2/e_Y**3
    d['v_T']     = d['r_o']*d['omega_o']        # transverse speed   [m/s]
    d['Delta_to']= (a/(beta*c))*zeta_closed(e, O)
    d['t_o']     = d['r_o']/c
    if kappa_R is not None:
        d['R_sur']  = R_s/kappa_R**2
        rho_body    = kappa_R**6*c**2/(8*PI*G*R_s**2)
        d['tau_D']  = mp.sqrt(PI/(G*rho_body))  # time-density invariant [s]
    return d

def scale_map(g, p, R_s, c, G):
    """The ENTIRE role of units: multiply pure numbers by monomial scale factors."""
    L, Tm = R_s, R_s/c                          # length unit, time unit
    return {
        'R_s':  R_s,                            # the anchor itself
        'a':    g['a_hat']*L,   'r_p': g['r_p_hat']*L, 'r_a': g['r_a_hat']*L,
        'omega': g['omega_hat']/Tm, 'T': g['T_hat']*Tm,
        'h_W':  g['h_hat']*L*c,
        'M':    g['M_hat']*c**2*R_s/G,          # geometric mass 1/2 -> kilograms
        'rho':  g['rho_hat']*c**2/(G*R_s**2),
        't':    g['t_hat']*Tm,
        'r_o':  p['r_o_hat']*L, 'omega_o': p['omega_o_hat']/Tm,
        'v_T':  p['beta_T']*c,
        'Delta_to': p['Delta_to_hat']*Tm, 't_o': p['t_o_hat']*Tm,
        'R_sur': g.get('R_sur_hat', mp.mpf(0))*L,
        'tau_D': g.get('tau_D_hat', mp.mpf(0))*Tm,
    }

N_ANCHORS = 40
for n in range(N_ANCHORS):
    beta, e = draw_state()
    O = rnd('0.02', str(float(2*mp.pi) - 0.02))
    kappa_R = rnd(str(float(mp.sqrt(2)*beta)*1.05), '0.99')
    # anchor: R_s drawn log-uniformly across 41 orders of magnitude [1e-15 m, 1e26 m]
    R_s = mp.mpf(10)**rnd('-15', '26')
    a   = R_s/(2*beta**2)
    M   = R_s*c_light**2/(2*G_newton)
    dim  = dimensional_pipeline(c_light, G_newton, M, a, e, O, kappa_R)
    g = core_global(beta, e, kappa_R=kappa_R)
    p = core_phase(beta, e, O)
    scl = scale_map(g, p, R_s, c_light, G_newton)
    for key in dim:
        check(2, f'commutation: {key}', dim[key], scl[key])

n2 = [r for r in RESULTS if r[0] == 2]
print(f"TEST 2: {sum(1 for r in n2 if r[2])}/{len(n2)} passed — dimensional pipeline == "
      f"unitless core x scale map, anchors spanning 1e-15..1e26 m "
      f"(worst error {mp.nstr(max(r[3] for r in n2), 3)})")

TEST 2: 680/680 passed — dimensional pipeline == unitless core x scale map, anchors spanning 1e-15..1e26 m (worst error 9.21e-61)


## TEST 3 — Unit-system invariance (including alien units)

The same physical system is processed in SI, CGS, imperial, astronomical and two
randomly generated **alien unit systems** (an alien civilisation's length tick and
time tick are irrational multiples of ours). The intrinsic solution is **bit-identical**
in every system, because its inputs are pure ratios that every observer in the
Universe measures as the same numbers. Only the final dimensional labels differ,
and they differ by exactly the unit conversion factors.

In [6]:
# ------------------------------------------------------------------
# Cell 6 — TEST 3: anthropocentric and alien unit systems
# ------------------------------------------------------------------
# Each system: (name, metres per length-unit, seconds per time-unit)
UNIT_SYSTEMS = [
    ('SI (m, s)',           mp.mpf(1),            mp.mpf(1)),
    ('CGS (cm, s)',         mp.mpf('0.01'),       mp.mpf(1)),
    ('imperial (ft, s)',    mp.mpf('0.3048'),     mp.mpf(1)),
    ('astronomical (au, d)',mp.mpf('1.495978707e11'), mp.mpf(86400)),
    ('alien-1 (phi*m, e*s)',(1+mp.sqrt(5))/2,     mp.e),
    ('alien-2 (random)',    mp.mpf(10)**rnd('-3','3')*mp.sqrt(2), mp.mpf(10)**rnd('-3','3')*mp.sqrt(3)),
]

beta_true, e_true = mp.mpf('0.1'), mp.mpf('0.5')   # reference system of Unitless_ROM.txt
T_SI = mp.mpf('1e6')*PI                            # measured period: 1e6 pi seconds

print(f"{'unit system':24s} {'beta (intrinsic)':>20s} {'a_hat':>8s} {'R_s in local units':>28s}")
core_prints = set()
for name, m_per_L, s_per_T in UNIT_SYSTEMS:
    # Step 0: the observables are RATIOS — identical numbers in every system.
    #         Only the anchor carries a unit: the period expressed in local ticks.
    T_local = T_SI / s_per_T                       # same duration, alien clock ticks
    c_local = c_light * s_per_T / m_per_L          # c expressed in local units
    # Steps 1-3: the intrinsic solution — no unit enters.
    g = core_global(beta_true, e_true)
    # Step 5: unit injection, local flavour.
    R_s_local = T_local * c_local * beta_true**3 / PI
    R_s_SI    = R_s_local * m_per_L
    check(3, f'{name}: R_s converts back to SI', R_s_SI, T_SI*c_light*beta_true**3/PI)
    fingerprint = mp.nstr(g['a_hat'], 50) + mp.nstr(g['Delta_phi'], 50) + mp.nstr(g['tau'], 50)
    core_prints.add(fingerprint)
    print(f"{name:24s} {mp.nstr(beta_true, 12):>20s} {mp.nstr(g['a_hat'], 6):>8s} "
          f"{mp.nstr(R_s_local, 12):>28s}")

check(3, 'intrinsic solution bit-identical across all unit systems',
      mp.mpf(len(core_prints)), 1)
# The reference values of Unitless_ROM.txt PHASE 1 (beta = 1/10, e = 1/2):
g_ref = core_global(beta_true, e_true)
check(3, 'reference: a_hat = 50', g_ref['a_hat'], 50)
check(3, 'reference: r_p_hat = 25', g_ref['r_p_hat'], 25)
check(3, 'reference: r_a_hat = 75', g_ref['r_a_hat'], 75)
check(3, 'reference: T_hat = 1000 pi', g_ref['T_hat'], 1000*PI)
check(3, 'reference: kappa_p = 0.2', g_ref['kappa_p'], mp.mpf('0.2'))
check(3, 'reference: Delta_phi = (149/1875) pi', g_ref['Delta_phi'], mp.mpf(149)/1875*PI)
check(3, 'reference: R_s anchor = 1e3 c (SI)', T_SI*c_light*beta_true**3/PI, 1000*c_light)

n3 = [r for r in RESULTS if r[0] == 3]
print(f"TEST 3: {sum(1 for r in n3 if r[2])}/{len(n3)} passed — one relational solution, "
      f"{len(UNIT_SYSTEMS)} unit dialects.")

unit system                  beta (intrinsic)    a_hat           R_s in local units
SI (m, s)                                 0.1     50.0               299792458000.0
CGS (cm, s)                               0.1     50.0               2.99792458e+13
imperial (ft, s)                          0.1     50.0               983571056430.0
astronomical (au, d)                      0.1     50.0                 2.0039888041
alien-1 (phi*m, e*s)                      0.1     50.0               185281928615.0
alien-2 (random)                          0.1     50.0                3807206573.57
TEST 3: 14/14 passed — one relational solution, 6 unit dialects.


## TEST 4 — $c$ and $G$ are translation factors, not physics inputs

$c$ and $G$ are multiplied by random factors (simulating different metrological
conventions / natural-unit choices). The intrinsic solution does not move by a single
digit. The dimensional outputs move **exactly** as the monomial scale map predicts:
$R_s \propto c$ (chronometric anchor), $M \propto c^3$ at fixed $T$... every exponent
is verified. A parameter whose only effect is a predictable monomial rescaling of
outputs is a unit convention, not a degree of freedom.

In [7]:
# ------------------------------------------------------------------
# Cell 7 — TEST 4: constant independence
# ------------------------------------------------------------------
beta0, e0 = mp.mpf('0.07'), mp.mpf('0.31')
T_anchor = mp.mpf('5.4321e7')                     # one clock reading [s]
g0 = core_global(beta0, e0)
p0 = core_phase(beta0, e0, mp.mpf('1.234'))
base_fp = mp.nstr(g0['Delta_phi'], 55) + mp.nstr(p0['beta_o'], 55)

for trial in range(12):
    fc = mp.mpf(10)**rnd('-2', '2')               # scramble c by up to 4 orders
    fG = mp.mpf(10)**rnd('-2', '2')               # scramble G by up to 4 orders
    c2, G2 = c_light*fc, G_newton*fG
    # intrinsic solution after scrambling the "constants": must be untouched
    g = core_global(beta0, e0)
    p = core_phase(beta0, e0, mp.mpf('1.234'))
    fp = mp.nstr(g['Delta_phi'], 55) + mp.nstr(p['beta_o'], 55)
    check(4, f'intrinsic state untouched by c*{mp.nstr(fc,3)}, G*{mp.nstr(fG,3)}',
          mp.mpf(1 if fp == base_fp else 0), 1)
    # dimensional outputs scale EXACTLY as the monomials of the scale map predict
    R_s_base = T_anchor*c_light*beta0**3/PI
    R_s_new  = T_anchor*c2*beta0**3/PI
    check(4, 'R_s responds as c^1 (chronometric anchor)', R_s_new/R_s_base, fc)
    M_base = R_s_base*c_light**2/(2*G_newton)
    M_new  = R_s_new*c2**2/(2*G2)
    check(4, 'M responds as c^3/G (pure unit translation)', M_new/M_base, fc**3/fG)
    rho_base = g['rho_hat']*c_light**2/(G_newton*R_s_base**2)
    rho_new  = g['rho_hat']*c2**2/(G2*R_s_new**2)
    check(4, 'rho responds as 1/G (at fixed clock anchor)', rho_new/rho_base, 1/fG)

n4 = [r for r in RESULTS if r[0] == 4]
print(f"TEST 4: {sum(1 for r in n4 if r[2])}/{len(n4)} passed — c, G confined to the scale map.")

TEST 4: 48/48 passed — c, G confined to the scale map.


## TEST 5 — All observational routes land on the same intrinsic state

The operational algorithm, executed end-to-end. A hidden ground-truth system emits
exact synthetic observables; six independent $\beta$-channels and four $e$-channels
invert them; every route must land on the same $(\beta, e)$ to 50 digits — and hence
on the same fully solved system.

In [8]:
# ------------------------------------------------------------------
# Cell 8 — TEST 5: route equivalence (the operational algorithm)
# ------------------------------------------------------------------
for trial in range(25):
    beta_true, e_true = draw_state()
    g = core_global(beta_true, e_true)

    # --- synthesize exact observables from the hidden ground truth ------------
    w_ratio   = g['e_X']**2                       # omega_max/omega_min (E1)
    bp, ba    = g['beta_p'], g['beta_a']          # apsidal Doppler pair (E2/B4)
    ra_rp     = g['r_a_hat']/g['r_p_hat']         # apsidal sky-angle ratio (E3)
    B_a_obs   = g['B_a']                          # balance-point phase (E4)
    z_b, z_k  = g['z_b'], g['z_k']                # spectroscopic pair (B1/B2)
    Z_sys     = g['Z_sys']                        # combined shift (B3)
    O1, O2 = mp.mpf('0.7'), mp.mpf('2.9')         # two-point route (B5)
    p1, p2 = core_phase(beta_true, e_true, O1), core_phase(beta_true, e_true, O2)
    r21 = p2['r_o_hat']/p1['r_o_hat']
    kappa_R = mp.sqrt(2)*beta_true*mp.mpf('1.7')  # surface route (B6)
    gs = core_global(beta_true, e_true, kappa_R=kappa_R)
    z_surf = 1/mp.sqrt(1 - kappa_R**2) - 1
    sin_th = gs['sin_theta_sys']

    # --- invert every channel ---------------------------------------------------
    for name, e_rec in [('E1 angular-speed ratio', e_from_omega_ratio(w_ratio)),
                        ('E2 apsidal Doppler',     e_from_apsidal_doppler(bp, ba)),
                        ('E3 apsidal radii',       e_from_apsidal_radii(ra_rp)),
                        ('E4 balance phase',       e_from_balance_phase(B_a_obs))]:
        check(5, f'{name} -> e', e_rec, e_true)
    for name, b_rec in [('B1 transverse Doppler',  beta_from_zb(z_b)),
                        ('B2 gravitational redshift', beta_from_zk(z_k)),
                        ('B3 combined Z_sys',      beta_from_Zsys(Z_sys)),
                        ('B4 apsidal Doppler',     beta_from_apsidal_doppler(bp, ba)),
                        ('B5 two-point vis-viva',  beta_from_two_point(r21, p1['beta_o'], p2['beta_o'])),
                        ('B6 surface channel',     beta_from_surface(z_surf, sin_th))]:
        check(5, f'{name} -> beta', b_rec, beta_true)

    # --- full parametrisation from one recovered pair must equal ground truth ---
    g_rec = core_global(beta_from_Zsys(Z_sys), e_from_omega_ratio(w_ratio))
    for key in ['a_hat', 'Delta_phi', 'T_hat', 'h_hat', 'kappa_p', 'Q']:
        check(5, f'recovered system: {key}', g_rec[key], g[key])

n5 = [r for r in RESULTS if r[0] == 5]
print(f"TEST 5: {sum(1 for r in n5 if r[2])}/{len(n5)} passed — 10 observational routes, "
      f"one intrinsic state.")

TEST 5: 400/400 passed — 10 observational routes, one intrinsic state.


## TEST 6 — Real data, last: Mercury and Jupiter

The documented four-observable pipeline ($\theta_\odot$, $z_{sun}$, $T_{ratio}$, $e$),
executed on published values. The entire system is solved **before** any unit appears;
the single anchor $T$ (one clock reading, in seconds because our civilisation's clock
ticks in seconds) then names $R_s$, $a$, $r_p$ in metres, and $G$ converts geometric
mass to kilograms in the very last line. Discrepancies sit within the observational
uncertainty of the inputs (±0.1%), as stated in the source document.

In [9]:
# ------------------------------------------------------------------
# Cell 9 — TEST 6: Mercury-Sun and Jupiter-Sun (application, data last)
# ------------------------------------------------------------------
theta_sun = mp.mpf('0.0046524')      # solar angular radius from Earth [rad]
z_sun     = mp.mpf('2.1224e-6')      # solar surface gravitational redshift
e_M       = mp.mpf('0.2056')         # Mercury eccentricity (angular-speed ratio)
T_ratio_M = mp.mpf('0.2408')         # Mercury/Earth cycle ratio
T_M       = mp.mpf('7600521.6')      # Mercury period [s] — THE dimensional anchor

# Stage 0-1: pure ratios -> intrinsic state (no unit anywhere)
R_ratio   = theta_sun / (T_ratio_M**mp.mpf('2/3') * (1 - e_M))
kappa_p2  = (1 - (1 + z_sun)**-2) * R_ratio
beta_p2   = kappa_p2 * (1 + e_M) / 2
beta2_M   = kappa_p2 - beta_p2                   # global invariant (Energy Symmetry)
beta_M    = mp.sqrt(beta2_M)

# Stage 2: solved system (still no unit)
gM = core_global(beta_M, e_M)
prec_rad  = gM['Delta_phi']                      # rad per orbit — a pure number

check(6, 'Mercury kappa_p^2 vs documented 6.4219e-8', kappa_p2, mp.mpf('6.4219e-8'),
      tol=mp.mpf('2e-4'))
check(6, 'Mercury beta^2 vs documented 2.5508e-8', beta2_M, mp.mpf('2.5508e-8'),
      tol=mp.mpf('2e-4'))
check(6, 'Mercury precession vs documented 5.0203e-7 rad/orbit', prec_rad,
      mp.mpf('5.0203e-7'), tol=mp.mpf('2e-4'))

# Anthropocentric bookkeeping: radians/orbit -> arcsec/century (calendar convention)
orbits_per_century = 100 * mp.mpf('365.25') / (T_M / 86400)
prec_arcsec = prec_rad * (180/PI) * 3600 * orbits_per_century
check(6, 'Mercury precession ~ 43 arcsec/century', prec_arcsec, mp.mpf('43.0'),
      tol=mp.mpf('2e-3'))

# Stage 5: unit injection — ONE clock reading names the scale
R_s_M = T_M * c_light * beta_M**3 / PI
a_M   = R_s_M / gM['kappa']**2
r_p_M = R_s_M / gM['kappa_p']**2
R_s_GR = mp.mpf('2953.33')                       # published 2GM_sun/c^2 [m]
check(6, 'R_s(Sun) vs GR value, within 0.1%', R_s_M, R_s_GR, tol=mp.mpf('1e-3'))
check(6, 'a(Mercury) vs 5.79e10 m, within 0.1%', a_M, mp.mpf('5.79e10'), tol=mp.mpf('1e-3'))
check(6, 'r_p(Mercury) vs 4.60e10 m, within 0.1%', r_p_M, mp.mpf('4.60e10'), tol=mp.mpf('1.5e-3'))

# Jupiter: same central invariants, different planet — universality check
e_J       = mp.mpf('0.04839266')
T_ratio_J = mp.mpf('11.862')
T_J       = mp.mpf('4332.589') * 86400           # [s] anchor
kappa_J2  = (1 - (1 + z_sun)**-2) * theta_sun / T_ratio_J**mp.mpf('2/3')
beta_J    = mp.sqrt(kappa_J2/2)
gJ = core_global(beta_J, e_J)
R_s_J = T_J * c_light * beta_J**3 / PI
a_J   = R_s_J / kappa_J2
check(6, 'R_s(Sun) recovered from Jupiter, within 0.15%', R_s_J, R_s_GR, tol=mp.mpf('1.5e-3'))
check(6, 'a(Jupiter) vs 7.7857e11 m, within 0.15%', a_J, mp.mpf('7.7857e11'), tol=mp.mpf('1.5e-3'))
check(6, 'same Sun from two planets: R_s(Mercury route)/R_s(Jupiter route) = 1',
      R_s_M/R_s_J, 1, tol=mp.mpf('2e-3'))

# Light deflection at the solar surface (grazing regime — formula in its domain)
kappa_R2_sun = 1 - (1 + z_sun)**-2
Delta_gamma = 2*mp.asin(kappa_R2_sun / (1 - kappa_R2_sun))
check(6, 'solar grazing light deflection ~ 1.75 arcsec',
      Delta_gamma*(180/PI)*3600, mp.mpf('1.751'), tol=mp.mpf('2e-3'))

# G enters HERE and only here: naming the geometric mass in kilograms
M_sun = R_s_M * c_light**2 / (2*G_newton)
check(6, 'M_sun vs 1.98847e30 kg (G used only as unit converter)', M_sun,
      mp.mpf('1.98847e30'), tol=mp.mpf('1e-3'))

print(f"TEST 6: Mercury: R_s = {mp.nstr(R_s_M, 6)} m, a = {mp.nstr(a_M, 7)} m, "
      f"precession = {mp.nstr(prec_arcsec, 5)} arcsec/century")
print(f"        Jupiter: R_s = {mp.nstr(R_s_J, 6)} m, a = {mp.nstr(a_J, 7)} m")
print(f"        M_sun = {mp.nstr(M_sun, 6)} kg  (G appears in this line only)")

TEST 6: Mercury: R_s = 2955.37 m, a = 5.792289e+10 m, precession = 43.001 arcsec/century
        Jupiter: R_s = 2954.8 m, a = 7.782173e+11 m
        M_sun = 1.98984e+30 kg  (G appears in this line only)


In [10]:
# ------------------------------------------------------------------
# Cell 10 — Scoreboard
# ------------------------------------------------------------------
BLOCKS = {1: 'TEST 1  intrinsic closure (pure numbers only)',
          2: 'TEST 2  solve-then-scale == dimensional pipeline',
          3: 'TEST 3  unit-system invariance (incl. alien units)',
          4: 'TEST 4  c, G confined to the scale map',
          5: 'TEST 5  observational route equivalence',
          6: 'TEST 6  Mercury + Jupiter application (data last)'}

print("=" * 78)
total_pass = total = 0
all_ok = True
for b in sorted(BLOCKS):
    rows = [r for r in RESULTS if r[0] == b]
    npass = sum(1 for r in rows if r[2])
    worst = max((r[3] for r in rows), default=mp.mpf(0))
    total_pass += npass; total += len(rows)
    status = 'PASS' if npass == len(rows) else 'FAIL'
    all_ok &= (npass == len(rows))
    print(f"{BLOCKS[b]:55s} {npass:5d}/{len(rows):<5d} {status}  worst {mp.nstr(worst, 3)}")
print("=" * 78)
print(f"{'TOTAL':55s} {total_pass:5d}/{total:<5d} {'ALL PASS' if all_ok else 'FAILURES PRESENT'}")
if not all_ok:
    print("\nFailed checks:")
    for r in RESULTS:
        if not r[2]:
            print(f"  [{r[0]}] {r[1]}  err = {mp.nstr(r[3], 5)}")
print()
print("CONCLUSION: the relational system is solved entirely in pure numbers;")
print("one dimensional anchor scales it into any civilisation's units; c and G")
print("never enter the solution — only the labelling. Units are the epilogue,")
print("not the physics.")

TEST 1  intrinsic closure (pure numbers only)            9501/9501  PASS  worst 3.91e-57
TEST 2  solve-then-scale == dimensional pipeline          680/680   PASS  worst 9.21e-61
TEST 3  unit-system invariance (incl. alien units)         14/14    PASS  worst 6.42e-61
TEST 4  c, G confined to the scale map                     48/48    PASS  worst 5.88e-61
TEST 5  observational route equivalence                   400/400   PASS  worst 4.52e-58
TEST 6  Mercury + Jupiter application (data last)          12/12    PASS  worst 0.000691
TOTAL                                                   10655/10655 ALL PASS

CONCLUSION: the relational system is solved entirely in pure numbers;
one dimensional anchor scales it into any civilisation's units; c and G
never enter the solution — only the labelling. Units are the epilogue,
not the physics.
